Fitting a linear probe, uses average pooling instead of a CLS token

In [1]:
import torch
from torch import nn, optim
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
from math import isqrt
import os
import json

In [2]:
from model import get_pos_embeddings, Net

In [3]:
MODEL_DIR = "checkpoints"
with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    cfg = json.load(f)

In [4]:
SEED = 12

In [5]:
# SSL TRAINING PARAMETERS

# --- Base params (from config) ---
# Image
D_IMAGE    = cfg["data"]["d_image"]
N_CHANNELS = cfg["data"]["n_channels"]
PATCH_SIZE = cfg["data"]["patch_size"]
# Encoder
D_ENC            = cfg["model"]["d_enc"]
N_HEADS_ENC      = cfg["model"]["n_heads_enc"]
N_ENCODER_BLOCKS = cfg["model"]["n_encoder_blocks"]
MLP_RATIO        = cfg["model"]["mlp_ratio"]
# Decoder
D_DEC            = cfg["model"]["d_dec"]
N_HEADS_DEC      = cfg["model"]["n_heads_dec"]
N_DECODER_BLOCKS = cfg["model"]["n_decoder_blocks"]
# Training
WORLD_SIZE            = cfg["metadata"]["world_size"]
LR                    = cfg["metadata"]["lr"]
BATCH_SIZE            = cfg["metadata"]["batch_size"]
WEIGHT_DECAY          = cfg["metadata"]["weight_decay"]
MOMENTUM              = cfg["metadata"]["momentum"]            # [0.9, 0.95]
WARMUP_EPOCHS         = cfg["metadata"]["warmup_epochs"]
N_EPOCHS              = cfg["metadata"]["n_epochs"]
PERCENT_UNMASKED      = cfg["metadata"]["percent_unmasked"]
USE_AUTOCAST          = cfg["metadata"]["use_autocast"]
USE_GRAD_ACCUMULATION = cfg["metadata"]["use_grad_accumulation"]
MICRO_BATCH_SIZE      = cfg["metadata"]["micro_batch_size"]
LOAD_MODEL            = cfg["metadata"]["load_model"]
R_MODEL_PATH          = cfg["metadata"]["r_model_path"]
CHECKPOINT_EVERY      = cfg["metadata"]["checkpoint_every"] # epochs between compressed checkpoints
PLOT_EVERY            = cfg["metadata"]["plot_every"] # epochs between loss-curve refreshes
OUTPUT_DIR            = cfg["metadata"]["output_dir"]
CHECKPOINT_DIR        = cfg["metadata"]["checkpoint_dir"]
USE_HF                = cfg["metadata"]["use_hf"]
HF_REPO               = cfg["metadata"]["hf_repo"]
HF_OUTPUT_DIR         = cfg["metadata"]["hf_output_dir"]
# --- Derived params (computed) ---
D_PATCH   = (PATCH_SIZE ** 2) * N_CHANNELS 
N_PATCHES = (D_IMAGE ** 2) // (PATCH_SIZE ** 2)
N_ROWS    = D_IMAGE // PATCH_SIZE
D_ENC_MLP = int(MLP_RATIO * D_ENC)
D_DEC_MLP = int(MLP_RATIO * D_DEC)

In [6]:
# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(D_IMAGE, scale=(0.75, 1.0)), # Larger crops for linear probe
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2604, 0.2566, 0.2713))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2603, 0.2566, 0.2713))
])

In [7]:
# Download and load datasets
supervised_trainset = torchvision.datasets.STL10(
    root='./data', split='train', download=True, transform=train_transform
)
testset = torchvision.datasets.STL10(
    root='./data', split='test', download=True, transform=test_transform
)
# DataLoaders
testloader = DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

val_size = 500 # 10%
train_size = len(supervised_trainset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    supervised_trainset, [train_size, val_size], generator=generator
)
supervised_trainloader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
supervised_valloader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

In [8]:
device = torch.device('cuda:0')

In [9]:
from model import get_pos_embeddings, Net
pos_embeddings_enc = get_pos_embeddings(D_ENC, N_PATCHES, N_ROWS)
pos_embeddings_dec = get_pos_embeddings(D_DEC, N_PATCHES, N_ROWS)

net = Net(
    n_encoder_blocks=N_ENCODER_BLOCKS,
    n_decoder_blocks=N_DECODER_BLOCKS,
    d_image=D_IMAGE,
    patch_size=PATCH_SIZE,
    d_patch=D_PATCH,
    n_patches=N_PATCHES,
    n_rows=N_ROWS,
    d_enc=D_ENC,
    d_enc_mlp=D_ENC_MLP,
    d_dec=D_DEC,
    d_dec_mlp=D_DEC_MLP,
    n_heads_enc=N_HEADS_ENC,
    n_heads_dec=N_HEADS_DEC,
    pos_embeddings_enc=pos_embeddings_enc,
    pos_embeddings_dec=pos_embeddings_dec,
    percent_unmasked=PERCENT_UNMASKED)

In [10]:
path = os.path.join(MODEL_DIR, "checkpoint_epoch_760.pt")
# path = os.path.join(MODEL_DIR, "checkpoint_epoch_300.pt")
sd = torch.load(path, map_location="cpu", weights_only=True)["model_state_dict"]
net.load_state_dict(sd)
net.to(device)

Net(
  (img2enc_projection): Linear(in_features=192, out_features=384, bias=True)
  (encoder_blocks): ModuleList(
    (0-11): 12 x ModuleDict(
      (norm_a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm_b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp_a): Linear(in_features=384, out_features=1536, bias=True)
      (mlp_b): Linear(in_features=1536, out_features=384, bias=True)
    )
  )
  (enc_terminal_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (enc2dec_projection): Linear(in_features=384, out_features=256, bias=True)
  (decoder_blocks): ModuleList(
    (0-7): 8 x ModuleDict(
      (norm_a): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (n

In [11]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR, ConstantLR

net.eval()
N_CLASSES = 10
N_EPOCHS = 100

# # Simple cosine decay
linear_probe = nn.Linear(D_ENC, N_CLASSES).to(device)
optimizer_probe = optim.AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=0.0)
scheduler = CosineAnnealingLR(optimizer_probe, T_max=N_EPOCHS)

# WARMUP_EPOCHS = 5
# HOLD_EPOCHS = 10
# COSINE_EPOCHS = N_EPOCHS - WARMUP_EPOCHS - HOLD_EPOCHS  # 35

# linear_probe = nn.Linear(D_ENC, N_CLASSES).to(device)
# optimizer_probe = optim.AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=0.0)

# warmup = LinearLR(optimizer_probe, start_factor=0.01, total_iters=WARMUP_EPOCHS)
# hold = ConstantLR(optimizer_probe, factor=1.0, total_iters=HOLD_EPOCHS)
# cosine = CosineAnnealingLR(optimizer_probe, T_max=COSINE_EPOCHS)
# scheduler = SequentialLR(
#     optimizer_probe,
#     [warmup, hold, cosine],
#     milestones=[WARMUP_EPOCHS, WARMUP_EPOCHS + HOLD_EPOCHS],
# )

criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    # --- train ---
    linear_probe.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        with torch.no_grad():
            embeddings, _, _, _ = net.encode(inputs)
            reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        loss = criterion(logits, labels)

        optimizer_probe.zero_grad()
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item()
        n_batches += 1

        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    # --- validate ---
    linear_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            embeddings, _, _, _ = net.encode(inputs)
            reps = embeddings.mean(dim=1)
            logits = linear_probe(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer_probe.param_groups[0]['lr']
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in linear_probe.state_dict().items()}

# load best-validation weights back into the probe
linear_probe.load_state_dict(best_state)
linear_probe.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

epoch   1 | train_loss=2.3554 | val_acc=0.1280 | lr=1.00e-03
epoch   2 | train_loss=2.2633 | val_acc=0.2020 | lr=9.99e-04
epoch   3 | train_loss=2.1878 | val_acc=0.3060 | lr=9.98e-04
epoch   4 | train_loss=2.1123 | val_acc=0.3000 | lr=9.96e-04
epoch   5 | train_loss=2.0515 | val_acc=0.3780 | lr=9.94e-04
epoch   6 | train_loss=2.0021 | val_acc=0.3620 | lr=9.91e-04
epoch   7 | train_loss=1.9482 | val_acc=0.4000 | lr=9.88e-04
epoch   8 | train_loss=1.8924 | val_acc=0.4120 | lr=9.84e-04
epoch   9 | train_loss=1.8567 | val_acc=0.4220 | lr=9.80e-04
epoch  10 | train_loss=1.8316 | val_acc=0.4260 | lr=9.76e-04
epoch  11 | train_loss=1.7832 | val_acc=0.4180 | lr=9.70e-04
epoch  12 | train_loss=1.7427 | val_acc=0.4420 | lr=9.65e-04
epoch  13 | train_loss=1.7337 | val_acc=0.4720 | lr=9.59e-04
epoch  14 | train_loss=1.7223 | val_acc=0.5160 | lr=9.52e-04
epoch  15 | train_loss=1.6718 | val_acc=0.5200 | lr=9.46e-04
epoch  16 | train_loss=1.6658 | val_acc=0.5420 | lr=9.38e-04
epoch  17 | train_loss=1

In [12]:
# Test the best linear probe
linear_probe.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, data in enumerate(testloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs)
        reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.5420
